# Incremental Training with MLJFlux

This demonstration is available as a Jupyter notebook or julia script
[here](https://github.com/FluxML/MLJFlux.jl/tree/dev/docs/src/common_workflows/incremental_training).

In this workflow example we explore how to incrementally train MLJFlux models.

In [1]:
using Pkg
PKG_ENV = joinpath(@__DIR__, "..", "..", "..")
Pkg.activate(PKG_ENV);
Pkg.instantiate();

  Activating project at `~/GoogleDrive/Julia/MLJ/MLJFlux/docs`


**Julia version** is assumed to be at least 1.12.

### Basic Imports

In [2]:
using MLJ               # Has MLJFlux models
using Flux              # For more flexibility
import Optimisers       # native Flux.jl optimisers no longer supported
using StableRNGs        # for reproducibility across Julia versions
stable_rng() = StableRNGs.StableRNG(123);

### Loading and Splitting the Data

In [3]:
iris = load_iris() # a named-tuple of vectors
y, X = unpack(iris, ==(:target), rng=stable_rng())
X = fmap(column-> Float32.(column), X) # Flux prefers Float32 data
(X_train, X_test), (y_train, y_test) = partition(
    (X, y), 0.8,
    multi = true,
    shuffle = true,
    rng=stable_rng(),
);

### Instantiating the model

Now let's construct our model. This follows a similar setup to the one followed in the
Quick Start.

In [4]:
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux
clf = NeuralNetworkClassifier(
    builder=MLJFlux.MLP(; hidden=(5,4), σ=Flux.relu),
    eta = 0.01,
    batch_size=8,
    epochs=10,
    rng=stable_rng(),
)

[ Info: For silent loading, specify `verbosity=0`. 
import MLJFlux ✔


NeuralNetworkClassifier(
  builder = MLP(
        hidden = (5, 4), 
        σ = NNlib.relu), 
  finaliser = NNlib.softmax, 
  optimiser = Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8), 
  eta = 0.01, 
  loss = Flux.Losses.crossentropy, 
  epochs = 10, 
  batch_size = 8, 
  lambda = 0.0, 
  alpha = 0.0, 
  rng = StableRNGs.LehmerRNG(state=0x000000000000000000000000000000f7), 
  optimiser_changes_trigger_retraining = false, 
  acceleration = CPU1{Nothing}(nothing), 
  embedding_dims = Dict{Symbol, Real}())

Instead of explicitly specifying an optimiser, we are going with the default (which is
`Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8)`) and specifying a learning rate
override of `eta=0.01` (equivalent to specifying `optImiser=Adam(0.01)`).

### Initial round of training

Now let's train the model. Calling fit! will automatically train it for 100 epochs as
specified above.

In [5]:
mach = machine(clf, X_train, y_train)
fit!(mach, verbosity=0)

trained Machine; caches model-specific representations of data
  model: NeuralNetworkClassifier(builder = MLP(hidden = (5, 4), …), …)
  args: 
    1:	Source @526 ⏎ ScientificTypesBase.Table{AbstractVector{ScientificTypesBase.Continuous}}
    2:	Source @777 ⏎ AbstractVector{ScientificTypesBase.Multiclass{3}}


Let's evaluate the training loss and validation accuracy

In [6]:
training_loss = cross_entropy(predict(mach, X_train), y_train)

0.22537291610526708

In [7]:
val_acc = accuracy(predict_mode(mach, X_test), y_test)

0.9333333333333333

Poor performance it seems.

### Incremental Training

Now let's train it for another 30 epochs at half the original learning rate. All we need
to do is change these hyperparameters and call `fit!` again. The learned
model parameters (weights and biases) are not re-initialized.

In [8]:
clf.eta = clf.eta*0.5
clf.epochs = clf.epochs + 30
fit!(mach, verbosity=2);

[ Info: Updating machine(NeuralNetworkClassifier(builder = MLP(hidden = (5, 4), …), …), …) (with warm restart if possible).
[ Info: Loss is 0.2183
[ Info: Loss is 0.1993
[ Info: Loss is 0.1842
[ Info: Loss is 0.1698
[ Info: Loss is 0.1572
[ Info: Loss is 0.146
[ Info: Loss is 0.1361
[ Info: Loss is 0.1274
[ Info: Loss is 0.1197
[ Info: Loss is 0.1128
[ Info: Loss is 0.1067
[ Info: Loss is 0.1013
[ Info: Loss is 0.09651
[ Info: Loss is 0.0922
[ Info: Loss is 0.08833
[ Info: Loss is 0.08486
[ Info: Loss is 0.08171
[ Info: Loss is 0.07886
[ Info: Loss is 0.07626
[ Info: Loss is 0.07389
[ Info: Loss is 0.07172
[ Info: Loss is 0.06972
[ Info: Loss is 0.06788
[ Info: Loss is 0.06618
[ Info: Loss is 0.0646
[ Info: Loss is 0.06313
[ Info: Loss is 0.06176
[ Info: Loss is 0.06049
[ Info: Loss is 0.05929
[ Info: Loss is 0.05816


If you'd rather changes to `eta` or `optimiser` force a cold restart, then set the
`optimiser_changes_trigger_retraining` hyperparameter to `true`. You might want do this
when comparing the effect of different learning rates on the final performance, using
MLJ's `TunedModel` wrapper, for example.

Let's evaluate the training loss and
validation accuracy

In [9]:
training_loss = cross_entropy(predict(mach, X_train), y_train)

0.05386519457061754

In [10]:
training_accuracy = accuracy(predict_mode(mach, X_test), y_test)

0.9

That's much better. If we are rather interested in resetting the model parameters before
fitting, we can do `fit(mach, force=true)`.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*